# DSPy Evaluation — Ministral-3-8B + BFS-Optimized Prompt

Evaluates `dspy_bfs_optimized.json` on the **poem_sentiment** test set.

Sections:
1. Load model
2. Reconstruct DSPy program & load optimized weights
3. Single-example deep dive (prompt/response inspection)
4. Full test-set inference
5. Metrics report (accuracy, per-class P/R/F1, confusion matrix)

In [ ]:
import torch
import dspy
from collections import Counter
from typing import Literal
from datasets import load_dataset
from transformers import AutoTokenizer, Mistral3ForConditionalGeneration

print(f"DSPy  : {dspy.__version__}")
print(f"CUDA  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU   : {torch.cuda.get_device_name(0)}")

## 1. Load Model

In [ ]:
MODEL_ID = "unsloth/Ministral-3-8B-Base-2512-unsloth-bnb-4bit"


class MinistralLocalLM(dspy.LM):
    def __init__(self, model_id: str = MODEL_ID, max_new_tokens: int = 64):
        self.model          = model_id
        self.model_type     = "text"
        self.max_new_tokens = max_new_tokens
        self.history        = []
        self.kwargs         = {"temperature": 0.0, "max_tokens": max_new_tokens, "n": 1}

        print("Loading tokenizer ...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        print("Loading model (NF4) ...")
        self.hf_model = Mistral3ForConditionalGeneration.from_pretrained(
            model_id, device_map="auto"
        )
        self.hf_model.config.use_cache = False
        self.hf_model.eval()
        print(f"Ready.  VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    @torch.inference_mode()
    def _complete(self, text: str, max_new_tokens: int) -> str:
        inputs     = self.tokenizer(text, return_tensors="pt").to(self.hf_model.device)
        prompt_len = inputs["input_ids"].shape[1]
        out_ids    = self.hf_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=self.tokenizer.eos_token_id,
        )
        return self.tokenizer.decode(out_ids[0][prompt_len:], skip_special_tokens=True)

    def _messages_to_text(self, messages: list) -> str:
        return "\n\n".join(m.get("content", "") for m in messages)

    def __call__(self, prompt=None, messages=None, n=1, max_tokens=None, **kwargs):
        text        = self._messages_to_text(messages) if messages else (prompt or "")
        n_toks      = max_tokens or self.max_new_tokens
        completions = [self._complete(text, n_toks) for _ in range(n)]
        self.history.append({"prompt": text, "response": completions})
        return completions

    def inspect_history(self, n: int = 1):
        for entry in self.history[-n:]:
            print("=" * 60 + " PROMPT")
            print(entry["prompt"])
            print("=" * 60 + " RAW RESPONSE")
            for c in entry["response"]:
                print(repr(c))
            print()


lm = MinistralLocalLM(MODEL_ID, max_new_tokens=64)
dspy.configure(lm=lm)

## 2. Reconstruct the DSPy Program and Load Optimized Weights

In [ ]:
import json, re

FALLBACK_DEMOS = [
    ("and the dead leaves lie huddled and still", "negative"),
    ("what light through yonder window breaks",   "positive"),
    ("the sun sets slowly in the west",           "no_impact"),
    ("a tender joy yet touched with pain",        "mixed"),
]
VALID_LABELS = {"negative", "positive", "no_impact", "mixed"}


def _get(demo, field):
    """Works whether demo is a dict (loaded from JSON) or a dspy.Example."""
    return demo[field] if isinstance(demo, dict) else getattr(demo, field)


class PoemSentiment(dspy.Signature):
    """Classify the emotional sentiment of a poem verse."""
    verse: str = dspy.InputField(desc="A line or verse from a poem")
    sentiment: Literal["negative", "positive", "no_impact", "mixed"] = dspy.OutputField(
        desc="One of: negative (grief/sorrow), positive (joy/hope), "
             "no_impact (neutral/descriptive), mixed (ambivalent)"
    )


class SentimentClassifier(dspy.Module):
    def __init__(self):
        self.predict = dspy.Predict(PoemSentiment)

    def _build_prompt(self, verse: str) -> str:
        lines = [
            "Classify the sentiment of each poem verse. "
            "Output only a JSON object with key 'label'. "
            "Valid values: negative, positive, no_impact, mixed.\n"
        ]
        demos = self.predict.demos or []
        examples = [(_get(d, "verse"), _get(d, "sentiment")) for d in demos] \
                   or FALLBACK_DEMOS
        for text, label in examples:
            lines.append(f'Verse: "{text}"')
            lines.append(f'Output: {{"label": "{label}"}}\n')
        lines.append(f'Verse: "{verse}"')
        lines.append('Output: {')
        return "\n".join(lines)

    def _parse(self, completion: str) -> str | None:
        text = ("{" + completion).strip()
        try:
            obj = json.loads(text if text.endswith("}") else text + "}")
            label = obj.get("label", "").lower()
            return label if label in VALID_LABELS else None
        except (json.JSONDecodeError, ValueError):
            pass
        m = re.search(r'"label"\s*:\s*"(\w+)"', text)
        if m:
            label = m.group(1).lower()
            return label if label in VALID_LABELS else None
        for label in VALID_LABELS:
            if label in completion.lower():
                return label
        return None

    def forward(self, verse: str) -> dspy.Prediction:
        prompt      = self._build_prompt(verse)
        completions = lm(prompt=prompt, max_tokens=32)
        raw         = completions[0] if completions else ""
        return dspy.Prediction(sentiment=self._parse(raw))


# ---- Load the optimized weights ----
optimized_bfs = SentimentClassifier()
optimized_bfs.load("dspy_optimized/dspy_bfs_optimized.json")

print("Loaded demos:")
for i, d in enumerate(optimized_bfs.predict.demos):
    print(f"  [{i}] {_get(d, 'verse')!r:55s}  ->  {_get(d, 'sentiment')}")

## 3. Single-Example Deep Dive
Run one example and print the **full prompt** DSPy sends to the model and the **raw text** it returns.

In [ ]:
PROBE_VERSE = "and the dead leaves lie huddled and still"  # expected: negative

# ---- A: with BFS-loaded demos (biased: 6x no_impact, 2x positive) ----
result_biased = optimized_bfs(verse=PROBE_VERSE)
print(f"[BFS demos — biased]  expected=negative  got={result_biased.sentiment!r}")

# ---- B: with FALLBACK_DEMOS (balanced: 1 per label) ----
baseline = SentimentClassifier()   # no demos loaded → uses FALLBACK_DEMOS
result_balanced = baseline(verse=PROBE_VERSE)
print(f"[Fallback demos — balanced]  expected=negative  got={result_balanced.sentiment!r}")

print()
print("RAW response with balanced demos:")
lm.inspect_history(n=1)

In [ ]:
# Try a few more probes across all 4 labels
PROBES = [
    ("and the dead leaves lie huddled and still",  "negative"),
    ("what light through yonder window breaks",     "positive"),
    ("the sun sets slowly in the west",             "no_impact"),
    ("a tender joy yet touched with pain",          "mixed"),
]

print(f"{'Verse':55s}  {'Expected':12s}  {'Predicted':12s}  OK?")
print("-" * 95)
for verse, expected in PROBES:
    pred = optimized_bfs(verse=verse)
    ok   = "✓" if pred.sentiment == expected else "✗"
    print(f"{verse!r:55s}  {expected:12s}  {str(pred.sentiment):12s}  {ok}")

print()
print("(Inspect last 4 raw prompts/responses to see what the model actually generated)")

In [ ]:
# Show raw prompt+response for all 4 probes above
lm.inspect_history(n=4)

## 4. Full Test Set Evaluation

In [ ]:
from sklearn import metrics as skmetrics

LABEL_MAP = {0: "negative", 1: "positive", 2: "no_impact", 3: "mixed"}
VALID_LABELS = set(LABEL_MAP.values())

ds = load_dataset("google-research-datasets/poem_sentiment")

def hf_to_dspy(split):
    return [
        dspy.Example(verse=row["verse_text"], sentiment=LABEL_MAP[row["label"]]).with_inputs("verse")
        for row in split
    ]

test_data = hf_to_dspy(ds["test"])
print(f"Test set: {len(test_data)} examples")
print(Counter(ex.sentiment for ex in test_data))

In [ ]:
y_true, y_pred = [], []

for ex in test_data:
    try:
        pred  = optimized_bfs(verse=ex.verse)
        label = (pred.sentiment or "").lower().strip()
        if label not in VALID_LABELS:
            label = ""
    except Exception:
        label = ""
    y_true.append(ex.sentiment)
    y_pred.append(label)

acc = skmetrics.accuracy_score(y_true, y_pred)
print(f"Test accuracy (dspy_bfs_optimized): {acc:.4f}\n")
print("Prediction distribution:", Counter(y_pred))
print("True distribution:      ", Counter(y_true))
print()
print(skmetrics.classification_report(
    y_true, y_pred,
    labels=list(LABEL_MAP.values()),
    zero_division=0,
))

## 5. Metrics Report

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

LABELS = list(LABEL_MAP.values())   # ["negative", "positive", "no_impact", "mixed"]

# ── 1. Summary table ─────────────────────────────────────────────────────────
report = skmetrics.classification_report(
    y_true, y_pred, labels=LABELS, zero_division=0, output_dict=True
)

print("=" * 55)
print(f"  Overall Accuracy : {skmetrics.accuracy_score(y_true, y_pred):.4f}")
print(f"  Macro F1         : {report['macro avg']['f1-score']:.4f}")
print(f"  Weighted F1      : {report['weighted avg']['f1-score']:.4f}")
parse_fail = sum(1 for p in y_pred if p == "")
print(f"  Parse failures   : {parse_fail} / {len(y_pred)}")
print("=" * 55)
print()
print(f"{'Class':>12}  {'Prec':>6}  {'Rec':>6}  {'F1':>6}  {'Support':>7}")
print("-" * 47)
for lbl in LABELS:
    r = report[lbl]
    print(f"{lbl:>12}  {r['precision']:>6.3f}  {r['recall']:>6.3f}  {r['f1-score']:>6.3f}  {int(r['support']):>7}")

# ── 2. Confusion matrix heatmap ───────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred, labels=LABELS)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
im = axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks(range(len(LABELS))); axes[0].set_xticklabels(LABELS, rotation=35, ha="right")
axes[0].set_yticks(range(len(LABELS))); axes[0].set_yticklabels(LABELS)
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
axes[0].set_title("Confusion Matrix (counts)")
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        axes[0].text(j, i, str(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im, ax=axes[0])

# ── 3. Per-class P / R / F1 bar chart ────────────────────────────────────────
x = np.arange(len(LABELS))
w = 0.25
prec = [report[l]["precision"] for l in LABELS]
rec  = [report[l]["recall"]    for l in LABELS]
f1   = [report[l]["f1-score"]  for l in LABELS]

axes[1].bar(x - w, prec, w, label="Precision", color="steelblue")
axes[1].bar(x,     rec,  w, label="Recall",    color="darkorange")
axes[1].bar(x + w, f1,   w, label="F1",        color="seagreen")
axes[1].set_xticks(x); axes[1].set_xticklabels(LABELS, rotation=20, ha="right")
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel("Score"); axes[1].set_title("Per-Class Precision / Recall / F1")
axes[1].legend(); axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()